# Part B: DPO on a small open-weight model (free Colab T4)

**Status: this notebook produces the only Part B results. Until it is run, Part B has no results.**

1. Runtime > Change runtime type > **T4 GPU** (free tier, no paid features).
2. Runtime > **Run all**.
3. The last cell downloads `dpo_results.json`. Put it at `results/part_b/dpo_results.json` in the repo.

Model: `HuggingFaceTB/SmolLM2-135M-Instruct` (Apache-2.0). Data: `Anthropic/hh-rlhf`, `helpful-base` subset (MIT).
The DPO loss is written by hand below. No paid service or API key is used.

In [ ]:
!pip -q install "transformers>=4.45,<5" "datasets>=2.20,<4" accelerate

In [ ]:
import copy, json, os, platform, subprocess, time
import numpy as np, torch, torch.nn.functional as F
import datasets, transformers
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

SMOKE = os.environ.get("DPO_SMOKE") == "1"  # CI wiring check only: never a result
MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
N_TRAIN, N_HELDOUT, N_TRAIN_EVAL = (16, 16, 16) if SMOKE else (2000, 500, 500)
EPOCHS = 1 if SMOKE else 2
SEEDS = [0] if SMOKE else [0, 1, 2]
BATCH, LR, BETA = 8, 1e-5, 0.1
MAX_PROMPT, MAX_RESPONSE = 192, 128
OUT = "dpo_results.smoke.json" if SMOKE else "dpo_results.json"

has_gpu = torch.cuda.is_available()
assert has_gpu or SMOKE, "No GPU: Runtime > Change runtime type > T4 GPU, then Run all."
device = torch.device("cuda" if has_gpu else "cpu")
gpu_name = torch.cuda.get_device_name(0) if has_gpu else None
print(device, gpu_name, "SMOKE" if SMOKE else "FULL RUN")

## Data

Each hh-rlhf record holds two full transcripts. The prompt is everything up to the last `Assistant:` turn.
Train pairs come from the dataset's `train` split, held-out pairs from its separate `test` split.

In [ ]:
SEP = "\n\nAssistant:"

def split_pair(row):
    c, r = row["chosen"], row["rejected"]
    i, j = c.rfind(SEP), r.rfind(SEP)
    if i < 0 or j < 0 or c[:i] != r[:j]:
        return None
    prompt, chosen, rejected = c[:i + len(SEP)], c[i + len(SEP):], r[j + len(SEP):]
    if not chosen.strip() or not rejected.strip() or chosen == rejected:
        return None
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

def load_pairs(split, n):
    ds = load_dataset("Anthropic/hh-rlhf", data_dir="helpful-base", split=split).shuffle(seed=1234)
    out = []
    for row in ds:
        p = split_pair(row)
        if p:
            out.append(p)
        if len(out) == n:
            break
    assert len(out) == n, (split, len(out))
    return out

train_pairs = load_pairs("train", N_TRAIN)
heldout_pairs = load_pairs("test", N_HELDOUT)
train_eval_pairs = train_pairs[:N_TRAIN_EVAL]
tok = AutoTokenizer.from_pretrained(MODEL)
PAD = tok.eos_token_id
print(len(train_pairs), len(heldout_pairs)); print(train_pairs[0]["prompt"][-200:])

## DPO loss

With $\pi$ the trained model and $\pi_{ref}$ a frozen copy, the implicit reward is
$\hat r(y) = \beta \log\frac{\pi(y|x)}{\pi_{ref}(y|x)}$ and the loss is

$$L = -\mathbb{E}\log\sigma\big(\beta[\log\tfrac{\pi(y_c|x)}{\pi_{ref}(y_c|x)} - \log\tfrac{\pi(y_r|x)}{\pi_{ref}(y_r|x)}]\big)$$

(Rafailov et al., 2023). $\log\pi(y|x)$ is the sum of token log-probabilities over the response only.

In [ ]:
def encode(prompt, response):
    p = tok(prompt, add_special_tokens=False).input_ids[-MAX_PROMPT:]
    r = tok(response, add_special_tokens=False).input_ids[:MAX_RESPONSE - 1] + [tok.eos_token_id]
    return p + r, len(p)

def collate(prompts, responses):
    enc = [encode(p, r) for p, r in zip(prompts, responses)]
    L = max(len(e[0]) for e in enc)
    ids = torch.full((len(enc), L), PAD); mask = torch.zeros((len(enc), L), dtype=torch.long)
    resp = torch.zeros((len(enc), L), dtype=torch.bool)
    for i, (x, n_p) in enumerate(enc):
        ids[i, :len(x)] = torch.tensor(x); mask[i, :len(x)] = 1; resp[i, n_p:len(x)] = True
    return ids.to(device), mask.to(device), resp.to(device)

def seq_logp(model, ids, mask, resp):
    """Sum of log p(token) over response tokens, one value per sequence."""
    logits = model(input_ids=ids, attention_mask=mask).logits[:, :-1]
    lp = F.log_softmax(logits, -1).gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1)
    return (lp * resp[:, 1:]).sum(-1)

def pair_logps(model, batch):
    p = [b["prompt"] for b in batch]
    return (seq_logp(model, *collate(p, [b["chosen"] for b in batch])),
            seq_logp(model, *collate(p, [b["rejected"] for b in batch])))

def dpo_loss(pi_c, pi_r, ref_c, ref_r, beta=BETA):
    return -F.logsigmoid(beta * ((pi_c - ref_c) - (pi_r - ref_r))).mean()

# Sanity checks on the loss before spending GPU time.
z = torch.zeros(4)
assert abs(dpo_loss(z, z, z, z).item() - np.log(2)) < 1e-6
assert dpo_loss(torch.ones(4) * 50, z, z, z) < dpo_loss(z, torch.ones(4) * 50, z, z)

## Metrics

- **Preference accuracy**: share of pairs where the model gives the chosen response a higher log-probability than the rejected one.
- **Log-ratio accuracy**: same, using the DPO implicit reward (trained minus reference log-probability).
- Baselines: the untrained reference model, and "the longer response wins" (no model).
- 95% bootstrap CI over pairs for each run, then the mean across seeds.

In [ ]:
@torch.no_grad()
def all_logps(model, pairs, bs=16):
    model.eval()
    c, r = [], []
    for i in range(0, len(pairs), bs):
        a, b = pair_logps(model, pairs[i:i + bs])
        c.append(a.float().cpu()); r.append(b.float().cpu())
    return torch.cat(c).numpy(), torch.cat(r).numpy()

def ci(correct, seed=0, n_boot=10000):
    v = np.asarray(correct, dtype=float)
    idx = np.random.default_rng(seed).integers(0, len(v), (n_boot, len(v)))
    lo, hi = np.quantile(v[idx].mean(1), [0.025, 0.975])
    return {"mean": float(v.mean()), "ci_low": float(lo), "ci_high": float(hi), "n": len(v)}

def length_baseline(pairs):
    return ci([len(p["chosen"]) > len(p["rejected"]) for p in pairs])

ref = AutoModelForCausalLM.from_pretrained(MODEL).to(device).eval()
for p in ref.parameters():
    p.requires_grad_(False)
ref_logps = {"train": all_logps(ref, train_eval_pairs), "heldout": all_logps(ref, heldout_pairs)}
sets = {"train": train_eval_pairs, "heldout": heldout_pairs}
reference = {k: ci(ref_logps[k][0] > ref_logps[k][1]) for k in sets}
length = {k: length_baseline(v) for k, v in sets.items()}
print("reference", reference); print("longer wins", length)

## Train

In [ ]:
def train_one(seed):
    torch.manual_seed(seed)
    model = copy.deepcopy(ref).train()
    for p in model.parameters():
        p.requires_grad_(True)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    rng, log, t0 = np.random.default_rng(seed), [], time.time()
    for epoch in range(EPOCHS):
        order = rng.permutation(len(train_pairs))
        for s in range(0, len(order) - BATCH + 1, BATCH):
            batch = [train_pairs[i] for i in order[s:s + BATCH]]
            with torch.no_grad():
                ref_c, ref_r = pair_logps(ref, batch)
            pi_c, pi_r = pair_logps(model, batch)
            loss = dpo_loss(pi_c, pi_r, ref_c, ref_r)
            opt.zero_grad(); loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            log.append({"loss": loss.item(), "grad_norm": float(grad_norm),
                        "margin": ((pi_c - ref_c) - (pi_r - ref_r)).mean().item()})
    minutes = (time.time() - t0) / 60
    row = {"seed": seed, "train_minutes": round(minutes, 2), "loss_curve": [round(x["loss"], 4) for x in log[::10]],
           "final_50_loss": float(np.mean([x["loss"] for x in log[-50:]])),
           "max_grad_norm": max(x["grad_norm"] for x in log)}
    for k, pairs in sets.items():
        c, r = all_logps(model, pairs)
        rc, rr = ref_logps[k]
        row[k] = {"preference_accuracy": ci(c > r), "log_ratio_accuracy": ci((c - rc) > (r - rr))}
    del model; torch.cuda.empty_cache() if has_gpu else None
    return row

runs = []
for seed in SEEDS:
    runs.append(train_one(seed)); print({k: v for k, v in runs[-1].items() if k != "loss_curve"})

## Save and download

In [ ]:
def across(metric, split):
    v = [r[split][metric]["mean"] for r in runs]
    return {"mean": float(np.mean(v)), "min": float(min(v)), "max": float(max(v)), "per_seed": v}

smi = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
results = {
    "smoke_run_not_a_result": SMOKE,
    "model": MODEL, "dataset": "Anthropic/hh-rlhf helpful-base (MIT)",
    "n_train": N_TRAIN, "n_heldout": N_HELDOUT, "n_train_eval": N_TRAIN_EVAL,
    "epochs": EPOCHS, "batch": BATCH, "lr": LR, "beta": BETA, "seeds": SEEDS,
    "hardware": {"gpu": gpu_name, "nvidia_smi": smi, "platform": platform.platform()},
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "datasets": datasets.__version__},
    "baselines": {"reference_model": reference, "longer_response_wins": length},
    "summary": {split: {m: across(m, split) for m in ("preference_accuracy", "log_ratio_accuracy")} for split in sets},
    "runs": runs,
}
json.dump(results, open(OUT, "w"), indent=1)
print(json.dumps({k: results[k] for k in ("baselines", "summary")}, indent=1))
if not SMOKE:
    from google.colab import files
    files.download(OUT)